# 50 SHADES OF AI — The Real-Estate Opportunity Swarm
## A 50-agent conversational decision architecture powered by Claude Sonnet 5

### Introduction

Buying real estate is one of the clearest examples of a decision that looks simple when represented on a website and becomes extraordinarily complex when experienced by a real person. A conventional property portal usually begins with a few explicit filters: location, maximum price, number of bedrooms, perhaps property type. Those variables are useful, but they describe only the visible surface of the decision. The actual problem is a multidimensional choice under uncertainty. A buyer must reconcile affordability with financing conditions; current family requirements with future family changes; commute with neighborhood quality; schools with taxes; architectural taste with maintenance; property age with renovation risk; emotional attachment with resale liquidity; and present value with an uncertain future market.

The difficulty becomes greater because the buyer's objective function is rarely known in advance. People can state preferences, but those preferences are incomplete, imprecise and sometimes internally inconsistent. A client may say that schools are the dominant concern, then repeatedly prefer homes in more walkable neighborhoods with somewhat weaker schools. Another may insist on contemporary architecture but become attracted to an older property because its location, lot and neighborhood character compensate for its age. A third may describe a fixed budget but discover that monthly carrying cost, rather than purchase price, is the real constraint. The task is therefore not merely to search a database. It is to infer what the buyer actually values, distinguish hard constraints from negotiable preferences, expose trade-offs, and continuously refine the mandate.

This notebook treats that problem as a natural application of a heterogeneous artificial-intelligence swarm. Instead of asking one monolithic model to declare which property is "best," we construct a society of **50 specialized agents**. The agents represent different perspectives that would normally be distributed across mortgage advisers, investment analysts, brokers, urban specialists, school researchers, architects, family advisers, risk managers and skeptical reviewers. Some agents focus on affordability and financing. Others examine resale liquidity, appreciation and optionality. Family agents care about schools, safety and usable space. Location agents examine commute, walkability and amenities. Lifestyle agents interpret architecture, condition and fit. Risk agents search for fragility. Governance agents ask whether an attractive narrative is violating a hard constraint or relying on weak evidence.

The purpose of heterogeneity is not theatrical. Disagreement is information. If every agent is instructed to think in the same way, a swarm is merely one model copied fifty times. Here, each specialist receives a distinct role and a distinct deterministic signal. The architecture then forms coalitions that aggregate related perspectives while preserving the dispersion of individual judgments. A property can therefore achieve a high overall score while still displaying meaningful dissent. That dissent becomes part of the recommendation rather than an error to be hidden.

Claude Sonnet 5 supplies the language-reasoning layer of the experiment. The notebook uses the current Anthropic model identifier `claude-sonnet-5` and reads the API credential from the Colab Secrets entry `ANTHROPIC_API_KEY`. The implementation deliberately does **not** send a `temperature` parameter. Structured responses are requested through Anthropic's structured-output capability when available, with a defensive fallback parser so that an API or SDK change does not destroy the entire notebook. Language-model reasoning is used where language is genuinely valuable: interpreting the buyer conversation, deliberating over competing candidate properties, identifying conflicts and producing an intelligible recommendation. Arithmetic, financial calculations, constraint enforcement, scoring validation and deterministic fallbacks remain outside the model. This separation is central to the design.

The notebook also separates the **data layer** from the **decision layer**. For reproducibility, it generates a synthetic universe of 1,200 properties distributed across synthetic neighborhoods. Each listing contains price, property type, style, bedrooms, bathrooms, square footage, lot size, age, condition, taxes, homeowners' association costs, commute, school quality, walkability, safety, amenities, noise, renovation requirements, days on market, price reductions, appreciation assumptions, rental yield, resale liquidity and climate risk. A production application would replace this synthetic adapter with authorized listing and public-data sources—MLS/RESO feeds, permitted market datasets, public property records, mortgage rates, school information, geospatial services, taxes, insurance and hazard data—without changing the swarm architecture.

The decision process proceeds in layers. First, conversation is converted into a structured **Buyer Mandate**. Second, deterministic financial engineering converts sticker prices into estimated all-in monthly carrying costs and forward scenarios. Third, the 50 agents evaluate the opportunity set. Fourth, hard constraints are enforced by a governance gate; the swarm cannot vote away the client's minimum bedrooms or maximum feasible budget simply because a house is attractive. Fifth, coalitions produce a consensus while the system measures dissent. Sixth, a Pareto frontier identifies properties that remain efficient across several competing objectives. Seventh, advocates and critics challenge the leading candidates. Eighth, the recommendation is subjected to interest-rate sensitivity. Finally, the entire process is exposed through a simple conversational application that runs inside Google Colab.

The result should not be interpreted as an automated real-estate adviser that replaces professional judgment. It is a research prototype for a more interesting proposition: **can a society of heterogeneous artificial specialists infer a human objective from conversation, search a complex opportunity space, preserve disagreement, enforce governance, and explain why a small set of opportunities deserves attention?**

Real estate is an unusually rich laboratory for this question because the objective is simultaneously financial and human. A home is an asset, a liability, a consumption good, a location, a family environment, an option on a neighborhood, and often an emotional object. Any system that reduces all of those dimensions to price per square foot has misunderstood the problem. The swarm developed here attempts something more ambitious: to transform a conversation into an evolving decision architecture and, ultimately, into an adaptive relationship in which the system learns from what the buyer actually chooses, rejects and revisits.

## Code Unit 1 — Environment, Claude Sonnet 5 and resilient structured reasoning

The first code unit establishes the computational and language-model environment on which the rest of the notebook depends. This is more consequential than a routine import cell because the architecture deliberately separates three forms of intelligence: deterministic computation, language interpretation and governance. NumPy and pandas handle the reproducible numerical state of the experiment; Plotly provides interpretable visualizations; Gradio supplies the lightweight application layer; and the Anthropic SDK connects the notebook to Claude Sonnet 5.

The API key is never written into the notebook. Instead, the cell attempts to read `ANTHROPIC_API_KEY` from Google Colab's Secrets facility. If the secret exists, `LIVE_LLM` becomes true and the language-reasoning components are activated. If it is absent, the notebook remains executable: later cells use deterministic fallbacks. This is useful pedagogically because a student can inspect the architecture without incurring API costs, while the same notebook becomes a live agentic experiment when the secret is available.

The model is explicitly set to `claude-sonnet-5`. No `temperature`, `top_p` or `top_k` parameter is supplied. The helper `claude_json` centralizes structured calls. It first attempts the current structured-output pathway using `output_config` with a JSON schema. If the installed SDK or endpoint rejects that particular structured-output signature, the helper records the failure and makes a conventional Messages API request whose prompt demands JSON, then validates the returned object locally. This fallback is intentionally defensive: the rest of the swarm should not be rewritten because an SDK surface changes.

Finally, the cell creates `LLM_LOG`, a provenance record containing the purpose, success or failure, model and error state of language-model calls. A serious autonomous system should never make invisible calls whose failures disappear. The log will later allow us to distinguish genuine Sonnet deliberation from deterministic fallback behavior.

In [ ]:
# @title
# CODE UNIT 1 — Environment, Claude Sonnet 5 and structured-output helper
!pip -q install -U anthropic gradio plotly pydantic

import os, json, re, math, random, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import gradio as gr
from pydantic import BaseModel, Field
from typing import List, Dict, Any, Optional, Literal

warnings.filterwarnings("ignore")
SEED = 766
rng = np.random.default_rng(SEED)
random.seed(SEED)
pd.set_option("display.max_columns", 100)

MODEL = "claude-sonnet-5"
LIVE_LLM = False
client = None
LLM_LOG = []

try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

try:
    import anthropic
    if ANTHROPIC_API_KEY:
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        LIVE_LLM = True
except Exception as e:
    LLM_LOG.append({"purpose":"client_init","ok":False,"error":repr(e)})

def _extract_text(message):
    parts=[]
    for block in getattr(message, "content", []):
        if getattr(block, "type", None) == "text":
            parts.append(block.text)
    return "\n".join(parts).strip()

def _json_from_text(text):
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.S)
        if not m:
            raise ValueError("No JSON object found in model response.")
        return json.loads(m.group(0))

def claude_json(purpose, system, prompt, schema, max_tokens=3000):
    """
    Sonnet 5 helper.
    - Uses model='claude-sonnet-5'
    - Intentionally sends NO temperature/top_p/top_k
    - First attempts Anthropic structured output via output_config.format
    - Falls back to a normal Messages call + local JSON validation if needed
    """
    if not LIVE_LLM or client is None:
        raise RuntimeError("LIVE_LLM is disabled or ANTHROPIC_API_KEY is unavailable.")

    structured_error = None
    try:
        message = client.messages.create(
            model=MODEL,
            max_tokens=max_tokens,
            system=system,
            messages=[{"role":"user","content":prompt}],
            output_config={
                "format":{
                    "type":"json_schema",
                    "schema":schema
                }
            }
        )
        data = _json_from_text(_extract_text(message))
        LLM_LOG.append({"purpose":purpose,"ok":True,"mode":"structured_output","model":MODEL})
        return data
    except Exception as e:
        structured_error = repr(e)

    fallback_prompt = (
        prompt
        + "\n\nReturn exactly one valid JSON object matching this JSON Schema. "
          "Do not add markdown fences or prose outside the JSON.\nSCHEMA:\n"
        + json.dumps(schema)
    )
    try:
        message = client.messages.create(
            model=MODEL,
            max_tokens=max_tokens,
            system=system,
            messages=[{"role":"user","content":fallback_prompt}]
        )
        data = _json_from_text(_extract_text(message))
        LLM_LOG.append({
            "purpose":purpose,"ok":True,"mode":"json_fallback","model":MODEL,
            "structured_error":structured_error
        })
        return data
    except Exception as e:
        LLM_LOG.append({
            "purpose":purpose,"ok":False,"model":MODEL,
            "structured_error":structured_error,"fallback_error":repr(e)
        })
        raise

print(f"MODEL={MODEL} | LIVE_LLM={LIVE_LLM}")
if not LIVE_LLM:
    print("Add ANTHROPIC_API_KEY to Colab Secrets to activate Claude Sonnet 5.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 50.8 MB/s eta 0:00:00
MODEL=claude-sonnet-5 | LIVE_LLM=True


## Code Unit 2 — Constructing a synthetic but economically coherent property market

The second unit creates the environment in which the swarm will make decisions. A toy table with ten randomly generated houses would be sufficient to demonstrate filtering, but it would be inadequate for demonstrating a complex decision architecture. The synthetic market therefore contains 1,200 properties and attempts to preserve economically meaningful correlations among variables. Neighborhoods have different latent profiles for prestige, schools, walkability, family suitability, culture, resale liquidity, transit and amenities. Property characteristics are then generated conditionally on those neighborhood attributes rather than independently.

This distinction matters. Real-estate variables are structurally related. More prestigious and walkable neighborhoods often command higher prices per square foot. Strong schools can contribute to both family demand and resale resilience. Condominiums typically differ from single-family homes in size, lot area and homeowners' association costs. Older properties may require more maintenance or renovation, while days on market and price reductions can provide imperfect signals of negotiation opportunity. Climate exposure can influence insurance costs. The simulation does not claim to reproduce any particular city; instead, it creates enough cross-sectional structure for specialists to disagree intelligently.

The database includes both immediately observable listing variables and derived economic signals. Among them are price, bedrooms, bathrooms, square footage, lot size, age, condition, annual tax, HOA charges, commute time, school score, walkability, safety, noise, renovation estimate, days on market, price-cut indicator, appreciation estimate, rental-yield estimate, resale liquidity and climate risk. These fields later support different agent perspectives.

Reproducibility is essential. A fixed random seed means that two readers running the notebook begin with the same synthetic world. Changes in recommendations can therefore be attributed to the buyer mandate, interest-rate assumptions or swarm logic rather than to a newly randomized market. The synthetic database is not the product; it is a controlled laboratory. In production, the same schema becomes the normalization target for authorized external feeds.

In [ ]:
# @title
# CODE UNIT 2 — Synthetic real-estate market and replaceable data adapter
N = 1200
neighborhoods = pd.DataFrame([
    ["Oak Park",.86,.91,.74,.82,.77,.83,.70,.82],
    ["River North",.96,.72,.93,.61,.91,.89,.88,.90],
    ["Maple Heights",.72,.88,.62,.91,.68,.74,.57,.73],
    ["Cedar Grove",.64,.79,.54,.87,.61,.68,.46,.66],
    ["Harbor Point",.89,.76,.85,.70,.88,.86,.81,.87],
    ["University Hill",.80,.94,.88,.67,.82,.84,.76,.85],
    ["Westfield",.69,.84,.67,.86,.72,.71,.60,.72],
    ["Old Town",.84,.78,.90,.63,.94,.80,.83,.84],
], columns=["neighborhood","prestige","schools","walkability","family","culture","resale","transit","amenities"])

styles = ["Modern","Contemporary","Traditional","Colonial","Craftsman","Mediterranean","Minimalist"]
types = ["Single Family","Townhouse","Condo"]
rows = []

for i in range(N):
    n = neighborhoods.iloc[rng.integers(len(neighborhoods))]
    typ = rng.choice(types,p=[.58,.20,.22])
    beds = int(np.clip(round(rng.normal(3.2 if typ!="Condo" else 2.1,1.05)),1,7))
    baths = float(np.clip(round(rng.normal(max(1.4,beds*.68),.55)*2)/2,1,6))
    sqft = int(np.clip(rng.normal(650+beds*520,420),550,6200))
    age = int(np.clip(rng.gamma(2,16),0,120))
    condition = float(np.clip(rng.normal(.78-age/420,.12),.25,1))
    lot = int(max(0,rng.normal(4800 if typ=="Single Family" else 900,2200)))
    psf = 180+330*n.prestige+100*n.walkability+70*n.schools
    price = max(120000,sqft*psf*(.80+.30*condition)*rng.lognormal(0,.12))
    tax = float(np.clip(rng.normal(.0125,.003),.006,.024))
    hoa = int(max(0,rng.normal(520 if typ=="Condo" else 160 if typ=="Townhouse" else 35,170)))
    commute = int(np.clip(52-31*n.transit+rng.normal(0,8),8,75))
    safety = float(np.clip(.55+.38*n.family+rng.normal(0,.08),0,1))
    noise = float(np.clip(.85*n.walkability+.35*n.culture-.50*n.family+rng.normal(0,.12),0,1))
    renovation = int(max(0,(1-condition)*price*rng.uniform(.05,.16)))
    dom = int(np.clip(rng.gamma(2,18)*(1.2-n.resale*.35),1,240))
    cut = float(np.clip((dom-25)/350+rng.normal(.02,.035),0,.18))
    appreciation = float(np.clip(.015+.045*n.resale+.012*n.schools+rng.normal(0,.012),-.02,.10))
    rental = float(np.clip(.025+.03*(1-n.prestige)+rng.normal(0,.008),.015,.085))
    climate = float(np.clip(rng.beta(2,6)+(.12 if n.neighborhood=="Harbor Point" else 0),0,1))
    rows.append([
        f"P{i+1:04d}",n.neighborhood,typ,rng.choice(styles),beds,baths,sqft,lot,age,condition,
        round(price,-3),tax,hoa,commute,n.schools,n.walkability,n.family,n.culture,n.amenities,
        safety,noise,renovation,dom,cut,appreciation,rental,n.resale,climate
    ])

COLS = ["property_id","neighborhood","property_type","style","beds","baths","sqft","lot_sqft","age",
        "condition","price","tax_rate","hoa_monthly","commute_min","schools","walkability","family_score",
        "culture","amenities","safety","noise","renovation_est","days_on_market","price_cut",
        "appreciation_est","rental_yield_est","resale_liquidity","climate_risk"]

properties = pd.DataFrame(rows,columns=COLS)
properties["annual_tax"] = properties.price*properties.tax_rate
properties["price_per_sqft"] = properties.price/properties.sqft

class PropertyDataAdapter:
    def search(self, mandate): raise NotImplementedError

class SyntheticAdapter(PropertyDataAdapter):
    def __init__(self,df): self.df=df.copy()
    def search(self,mandate): return self.df.copy()

class ProductionAdapterBlueprint(PropertyDataAdapter):
    """
    Replace with authorized production sources and normalize to COLS.
    Potential layers: MLS/RESO, public property records, permitted market data,
    schools, maps/travel time, mortgage rates, taxes, insurance and hazard data.
    """
    def search(self,mandate):
        raise NotImplementedError("Configure authorized production data sources.")

DATA = SyntheticAdapter(properties)
display(properties.head())
print("Synthetic universe:",properties.shape)

,property_id,neighborhood,property_type,style,beds,baths,sqft,lot_sqft,age,condition,price,tax_rate,hoa_monthly,commute_min,schools,walkability,family_score,culture,amenities,safety,noise,renovation_est,days_on_market,price_cut,appreciation_est,rental_yield_est,resale_liquidity,climate_risk,annual_tax,price_per_sqft
0,P0001,Cedar Grove,Single Family,Mediterranean,2,1.0,1403,9057,47,0.559459,705000.0,0.015388,66,42,0.79,0.54,0.87,0.61,0.66,0.873836,0.159318,25522,63,0.112558,0.070816,0.041649,0.68,0.249039,10848.594614,502.494654
1,P0002,University Hill,Single Family,Craftsman,5,3.5,3010,4405,27,0.542259,1754000.0,0.012398,382,24,0.94,0.88,0.67,0.82,0.85,0.789843,0.594775,70926,29,0.010429,0.077946,0.022166,0.84,0.235656,21746.763791,582.724252
2,P0003,University Hill,Single Family,Mediterranean,2,1.0,2288,1578,60,0.620449,1486000.0,0.012633,45,33,0.94,0.88,0.67,0.82,0.85,0.717332,0.641662,87791,62,0.105558,0.053558,0.037021,0.84,0.045813,18772.022636,649.475524
3,P0004,University Hill,Townhouse,Modern,1,1.5,1144,0,48,0.804210,701000.0,0.008947,157,29,0.94,0.88,0.67,0.82,0.85,0.758042,0.500839,17408,54,0.106127,0.059601,0.027241,0.84,0.202636,6271.553952,612.762238
4,P0005,River North,Townhouse,Mediterranean,3,1.5,2536,52,17,0.760495,1543000.0,0.011792,0,28,0.72,0.93,0.61,0.91,0.90,0.706827,0.610911,40166,12,0.000000,0.054310,0.040170,0.89,0.071681,18195.093962,608.438486


Synthetic universe: (1200, 30)


## Code Unit 3 — From human conversation to a formal Buyer Mandate

A real buyer does not naturally communicate through database fields. The buyer says, for example, that the children are approaching school age, that one spouse works from home, that the family dislikes long commutes, that a renovation would be acceptable only if the price compensates for it, or that resale matters because an international relocation is possible. Code Unit 3 therefore addresses the semantic problem at the center of the application: converting an ordinary conversation into a formal decision contract.

The `BuyerMandate` schema defines the variables that the downstream system can reason about. It includes hard or quasi-hard constraints—budget, minimum bedrooms, minimum bathrooms, minimum area and preferred maximum commute—as well as priorities such as schools, safety, walkability, resale, condition, architectural style, space, investment characteristics and climate risk. It also captures financing assumptions, renovation tolerance and expected holding horizon.

When `LIVE_LLM` is active, Claude Sonnet 5 receives the conversation and a JSON schema. Its task is not to recommend a house. It is to interpret the client's language conservatively and return a structured mandate. This separation prevents the interview model from silently jumping from ambiguous language to a property recommendation. The mandate can be displayed, inspected and corrected.

The cell also implements a deterministic parser. It recognizes common expressions for budget, bedrooms, bathrooms, family size, commute and holding period, and it raises priorities when corresponding concepts appear in the conversation. This parser is deliberately modest; it is not intended to compete with Sonnet's semantic interpretation. Its purpose is resilience and reproducibility.

Most importantly, the mandate is treated as provisional. Human preferences are latent. The first conversation supplies an initial hypothesis about the buyer's utility function, not an immutable truth. A production version should update this object as the client saves, rejects, tours and revisits properties. In that sense, the Buyer Mandate is the system's evolving model of the client.

In [ ]:
# @title
# CODE UNIT 3 — Conversation → Buyer Mandate using Sonnet 5 with deterministic fallback
DEFAULT_MANDATE = {
    "budget":900000.0,"down_payment_pct":.25,"mortgage_rate":.0625,"term_years":30,
    "max_monthly_housing":6500.0,"min_beds":3,"min_baths":2.0,"min_sqft":1500,
    "family_size":4,"school_priority":.8,"commute_priority":.6,"max_commute":40,
    "walkability_priority":.5,"resale_priority":.8,"condition_priority":.6,
    "style_priority":.5,"preferred_styles":["Modern","Contemporary","Craftsman"],
    "safety_priority":.8,"space_priority":.7,"investment_priority":.7,
    "culture_priority":.4,"climate_priority":.5,"renovation_tolerance":.35,
    "risk_tolerance":.5,"horizon_years":7
}

MANDATE_SCHEMA = {
    "type":"object",
    "properties":{
        "budget":{"type":"number"},"down_payment_pct":{"type":"number"},"mortgage_rate":{"type":"number"},
        "term_years":{"type":"integer"},"max_monthly_housing":{"type":"number"},"min_beds":{"type":"integer"},
        "min_baths":{"type":"number"},"min_sqft":{"type":"integer"},"family_size":{"type":"integer"},
        "school_priority":{"type":"number"},"commute_priority":{"type":"number"},"max_commute":{"type":"integer"},
        "walkability_priority":{"type":"number"},"resale_priority":{"type":"number"},"condition_priority":{"type":"number"},
        "style_priority":{"type":"number"},"preferred_styles":{"type":"array","items":{"type":"string"}},
        "safety_priority":{"type":"number"},"space_priority":{"type":"number"},"investment_priority":{"type":"number"},
        "culture_priority":{"type":"number"},"climate_priority":{"type":"number"},"renovation_tolerance":{"type":"number"},
        "risk_tolerance":{"type":"number"},"horizon_years":{"type":"integer"}
    },
    "required":list(DEFAULT_MANDATE.keys()),
    "additionalProperties":False
}

def deterministic_parse(text):
    m=DEFAULT_MANDATE.copy(); t=text.lower().replace(",","")
    vals=[]
    for num,unit in re.findall(r"\$?\s*(\d+(?:\.\d+)?)\s*(million|m|k|thousand)?",t):
        v=float(num)
        if unit in ["million","m"]: v*=1_000_000
        elif unit in ["k","thousand"]: v*=1_000
        if v>=100000: vals.append(v)
    if vals: m["budget"]=max(vals)
    patterns={
        "min_beds":[r"(\d+)\s*(?:bed|bedroom)"],
        "min_baths":[r"(\d+(?:\.\d+)?)\s*(?:bath|bathroom)"],
        "family_size":[r"family\s+of\s+(\d+)",r"(\d+)\s+people"],
        "max_commute":[r"commute.*?(\d+)\s*(?:min|minute)"],
        "horizon_years":[r"(\d+)\s*year.*?(?:hold|horizon|stay)"]
    }
    for key,ps in patterns.items():
        for p in ps:
            z=re.search(p,t)
            if z:
                m[key]=float(z.group(1)) if "." in z.group(1) else int(z.group(1)); break
    for words,key in [
        (["school","children","kids"],"school_priority"),(["resale","sell later"],"resale_priority"),
        (["walkable","walking"],"walkability_priority"),(["safe","safety"],"safety_priority"),
        (["commute","office"],"commute_priority"),(["investment","return","appreciation"],"investment_priority"),
        (["space","large","roomy"],"space_priority"),(["move-in","turnkey"],"condition_priority"),
        (["climate","flood","fire"],"climate_priority")]:
        if any(w in t for w in words): m[key]=.95
    if any(w in t for w in ["renovate","fixer","remodel"]): m["renovation_tolerance"]=.85
    if "modern" in t: m["preferred_styles"]=["Modern","Contemporary","Minimalist"]
    if "traditional" in t: m["preferred_styles"]=["Traditional","Colonial","Craftsman"]
    return m

def validate_mandate(m):
    out=DEFAULT_MANDATE.copy()
    for k in out:
        if k in m and m[k] is not None: out[k]=m[k]
    for k in ["school_priority","commute_priority","walkability_priority","resale_priority","condition_priority",
              "style_priority","safety_priority","space_priority","investment_priority","culture_priority",
              "climate_priority","renovation_tolerance","risk_tolerance"]:
        out[k]=float(np.clip(out[k],0,1))
    return out

def interpret_buyer(text):
    if LIVE_LLM:
        try:
            system=("You are the mandate-intake specialist for a governed real-estate buyer advisory system. "
                    "Infer conservatively. Distinguish stated facts from priorities. Do not recommend properties.")
            prompt=(
                "<buyer_conversation>\n"+text+"\n</buyer_conversation>\n"
                "Return the complete Buyer Mandate. Use defaults when the buyer gives no evidence to change them. "
                "Priorities are numbers from 0 to 1. Budget is purchase price, not monthly payment."
            )
            return validate_mandate(claude_json("buyer_mandate",system,prompt,MANDATE_SCHEMA,2500)), "Claude Sonnet 5"
        except Exception:
            pass
    return validate_mandate(deterministic_parse(text)), "deterministic fallback"

sample_interview = (
    "We are a family of 4 looking for at least 3 bedrooms and 2 baths. Budget around $950k. "
    "Schools and safety matter a lot. I work downtown and prefer a commute under 35 minutes. "
    "We like modern or contemporary homes, want reasonable space, and expect to stay about 7 years, "
    "so resale matters. We can renovate a little but do not want a major project."
)
mandate, mandate_mode = interpret_buyer(sample_interview)
print("Mandate mode:",mandate_mode)
display(pd.Series(mandate,name="value").to_frame())

Mandate mode: Claude Sonnet 5


,value
budget,950000
down_payment_pct,0.2
mortgage_rate,0.065
term_years,30
max_monthly_housing,6000
min_beds,3
min_baths,2
min_sqft,1500
family_size,4
school_priority,0.8


## Code Unit 4 — Financial engineering and the society of 50 heterogeneous specialists

The fourth unit performs two jobs that belong together conceptually. First, it converts property prices into economically meaningful carrying costs. Second, it defines the society of specialists who will interpret those economics and the nonfinancial attributes from different perspectives.

Purchase price is not affordability. The financing engine estimates mortgage payments from the down-payment assumption, mortgage rate and term. It then adds property taxes, a synthetic insurance proxy, maintenance and HOA costs to estimate an all-in monthly burden. It also calculates cash required at closing and a forward property value under the synthetic appreciation assumption. These calculations are deterministic. Claude is not asked to perform mortgage arithmetic because language models should not replace transparent formulas where exact formulas exist.

The second half of the unit creates 50 named agents divided into seven coalitions: Finance, Investment, Family, Location, Lifestyle, Risk and Governance. The names are intentionally specific. An Affordability agent is not identical to an Interest-Rate Risk agent; a Schools agent should not reason like a Child Friendliness agent; an Architecture specialist should not have the same objective as a Resale Liquidity specialist. The purpose is to create different informational lenses.

This is a key design principle for swarm systems. Merely asking the same model fifty times for an opinion does not create meaningful collective intelligence. Diversity must enter through roles, evidence, objective functions or information sets. In this notebook, diversity enters through specialist identities and deterministic feature maps, and later through Claude's coalition-level deliberation.

The architecture also avoids an unnecessary cost explosion. We retain 50 distinct agent scores, but we do not require 50 independent large-model calls for every search. Language reasoning is concentrated where it adds value: mandate interpretation and coalition deliberation over a compact shortlist. The swarm therefore remains computationally and economically plausible while preserving its heterogeneous structure.

In [ ]:
# @title
# CODE UNIT 4 — Financial engine + 50-agent heterogeneous society
def mortgage_payment(principal, annual_rate, years):
    r=annual_rate/12; n=years*12
    if r==0: return principal/n
    return principal*r*(1+r)**n/((1+r)**n-1)

def enrich_financials(df,m):
    x=df.copy()
    loan=x.price*(1-m["down_payment_pct"])
    x["mortgage_monthly"]=[mortgage_payment(p,m["mortgage_rate"],m["term_years"]) for p in loan]
    x["tax_monthly"]=x.annual_tax/12
    x["insurance_monthly"]=x.price*(.0035+.004*x.climate_risk)/12
    x["maintenance_monthly"]=x.price*(.006+.006*(1-x.condition))/12
    x["all_in_monthly"]=x.mortgage_monthly+x.tax_monthly+x.insurance_monthly+x.maintenance_monthly+x.hoa_monthly
    x["cash_at_close"]=x.price*m["down_payment_pct"]+x.renovation_est
    x["future_value_est"]=x.price*((1+x.appreciation_est)**m["horizon_years"])
    return x

agent_specs=[
("Affordability","Finance"),("Mortgage Structure","Finance"),("Interest-Rate Risk","Finance"),("Taxes","Finance"),
("Insurance Cost","Finance"),("Maintenance Burden","Finance"),("Renovation Budget","Finance"),("Total Cost of Ownership","Finance"),
("Resale Liquidity","Investment"),("Appreciation","Investment"),("Rental Optionality","Investment"),("Downside Risk","Investment"),
("Negotiation Opportunity","Investment"),("Market Timing","Investment"),("Price per SqFt","Investment"),("Neighborhood Momentum","Investment"),
("Schools","Family"),("Safety","Family"),("Family Space","Family"),("Bedrooms","Family"),("Bathrooms","Family"),
("Outdoor Space","Family"),("Child Friendliness","Family"),("Multigenerational Flexibility","Family"),
("Commute","Location"),("Transit","Location"),("Walkability","Location"),("Amenities","Location"),("Culture","Location"),
("Noise","Location"),("Accessibility","Location"),("Neighborhood Fit","Location"),
("Architecture","Lifestyle"),("Style Match","Lifestyle"),("Condition","Lifestyle"),("Move-in Readiness","Lifestyle"),
("Privacy","Lifestyle"),("Entertaining","Lifestyle"),("Work-from-Home","Lifestyle"),("Emotional Fit","Lifestyle"),
("Climate Risk","Risk"),("Property Age Risk","Risk"),("Liquidity Stress","Risk"),("Budget Stress","Risk"),
("Renovation Execution","Risk"),("Concentration Risk","Risk"),
("Data Quality","Governance"),("Constraint Auditor","Governance"),("Devil's Advocate","Governance"),("Buyer Advocate","Governance")
]
agents=pd.DataFrame(agent_specs,columns=["agent","coalition"])
agents["agent_id"]=[f"A{i+1:02d}" for i in range(50)]

market=enrich_financials(DATA.search(mandate),mandate)
display(agents.groupby("coalition").size().rename("number_of_agents"))
display(market[["property_id","price","mortgage_monthly","all_in_monthly","cash_at_close","future_value_est"]].head())

,number_of_agents
coalition,
Family,8
Finance,8
Governance,4
Investment,8
Lifestyle,8
Location,8
Risk,6


,property_id,price,mortgage_monthly,all_in_monthly,cash_at_close,future_value_est
0,P0001,705000.0,3564.863653,5306.853120,166522.0,1.138134e+06
1,P0002,1754000.0,8869.178506,12991.211662,421726.0,2.966256e+06
2,P0003,1486000.0,7514.024663,10604.476191,384991.0,2.141054e+06
3,P0004,701000.0,3544.637476,4895.199053,157608.0,1.051267e+06
4,P0005,1543000.0,7802.247682,10761.693433,348766.0,2.234318e+06


## Code Unit 5 — Specialist scoring, governance gates and the first collective ranking

This unit is where the 50-agent society begins to behave as a decision system. Each property is transformed into normalized features such as affordability, monthly-cost fit, school quality, safety, commute, space, resale liquidity, appreciation, rental optionality, negotiation opportunity, condition, age resilience, climate resilience, noise, amenities, cultural fit, architectural-style match and renovation burden. The feature transformations are transparent and bounded so that the resulting scores remain interpretable.

Each specialist then receives a different combination of those signals. Coalition membership determines the broad information set, while the agent's name increases the weight of the feature most closely related to its expertise. Buyer priorities modify these weights but do not erase specialist independence. This distinction is important. If the buyer says schools are extremely important, the Schools agent should become more influential, but a Climate Risk agent should not suddenly stop caring about climate.

A governance gate is applied independently of the popularity contest. Minimum bedrooms, bathrooms, area and budget are treated as hard constraints, while monthly carrying cost and commute are monitored as softer feasibility conditions. A property that violates a hard constraint receives a substantial penalty even if many lifestyle agents like it. This is the computational analogue of institutional governance: attractive narratives cannot simply vote away the mandate.

Agent scores are then aggregated into coalition scores and finally into an initial consensus. The system simultaneously computes the standard deviation across the 50 agents. That dispersion is labeled disagreement rather than noise. A high-consensus, low-disagreement property is qualitatively different from a property that achieves the same average score because half the swarm loves it and half strongly objects.

The output of this unit is the first auditable opportunity ranking. It is deliberately not yet the final answer. The next units will identify efficient trade-offs, expose advocates and critics, and invite Claude Sonnet 5 to deliberate over the most promising candidates.

In [ ]:
# @title
# CODE UNIT 5 — 50 specialist scores, governance, coalitions, consensus and dissent
def norm01(s,inverse=False):
    s=pd.Series(s,dtype=float)
    z=(s-s.min())/(s.max()-s.min()+1e-9)
    return 1-z if inverse else z

def base_features(df,m):
    f=pd.DataFrame(index=df.index)
    f["affordability"]=np.clip(1-df.price/m["budget"],-1,1)
    f["monthly"]=np.clip(1-df.all_in_monthly/m["max_monthly_housing"],-1,1)
    f["condition"]=df.condition; f["schools"]=df.schools; f["safety"]=df.safety
    f["walkability"]=df.walkability; f["commute"]=np.clip(1-df.commute_min/max(1,m["max_commute"]),-1,1)
    f["space"]=norm01(df.sqft); f["beds"]=np.clip(df.beds/max(1,m["min_beds"]),0,1.25)
    f["baths"]=np.clip(df.baths/max(1,m["min_baths"]),0,1.25); f["resale"]=df.resale_liquidity
    f["appreciation"]=norm01(df.appreciation_est); f["rental"]=norm01(df.rental_yield_est)
    f["negotiation"]=np.clip(df.price_cut*4+norm01(df.days_on_market)*.35,0,1)
    f["age"]=norm01(df.age,True); f["climate"]=1-df.climate_risk; f["noise"]=1-df.noise
    f["amenities"]=df.amenities; f["culture"]=df.culture
    f["style"]=df["style"].isin(m["preferred_styles"]).astype(float)
    f["lot"]=norm01(df.lot_sqft); f["renovation"]=1-norm01(df.renovation_est)
    return f

MAPS={
"Finance":["affordability","monthly","condition","renovation"],
"Investment":["resale","appreciation","rental","negotiation","affordability"],
"Family":["schools","safety","space","beds","baths","lot"],
"Location":["commute","walkability","amenities","culture","noise","safety"],
"Lifestyle":["style","condition","space","lot","culture","noise"],
"Risk":["climate","age","condition","resale","affordability","monthly"],
"Governance":["affordability","monthly","schools","safety","resale","condition","climate"]
}
KEYWORD={"school":"schools","safety":"safety","commute":"commute","walk":"walkability","resale":"resale",
"appreciation":"appreciation","rental":"rental","climate":"climate","style":"style","architecture":"style",
"condition":"condition","renovation":"renovation","noise":"noise","space":"space","bedroom":"beds",
"bathroom":"baths","amenit":"amenities","culture":"culture","age":"age","mortgage":"monthly",
"afford":"affordability","tax":"monthly","insurance":"monthly","liquidity":"resale","negotiation":"negotiation"}

def score_agent(name,df,m):
    f=base_features(df,m)
    coalition=agents.loc[agents.agent==name,"coalition"].iloc[0]
    w={c:1.0 for c in MAPS[coalition]}
    for k,v in KEYWORD.items():
        if k in name.lower(): w[v]=w.get(v,0)+3.0
    priority={"schools":m["school_priority"],"safety":m["safety_priority"],"commute":m["commute_priority"],
              "walkability":m["walkability_priority"],"resale":m["resale_priority"],
              "condition":m["condition_priority"],"style":m["style_priority"],"space":m["space_priority"],
              "appreciation":m["investment_priority"],"climate":m["climate_priority"]}
    for c in w: w[c]*=(.55+.9*priority.get(c,.6))
    return sum(f[c]*v for c,v in w.items())/sum(w.values())

def governance(df,m):
    g=pd.DataFrame(index=df.index)
    g["budget_ok"]=df.price<=m["budget"]*1.03
    g["beds_ok"]=df.beds>=m["min_beds"]; g["baths_ok"]=df.baths>=m["min_baths"]
    g["sqft_ok"]=df.sqft>=m["min_sqft"]
    g["monthly_ok"]=df.all_in_monthly<=m["max_monthly_housing"]*1.08
    g["commute_ok"]=df.commute_min<=m["max_commute"]*1.20
    g["hard_pass"]=g[["budget_ok","beds_ok","baths_ok","sqft_ok"]].all(axis=1)
    g["violation_count"]=(~g.drop(columns=["hard_pass"])).sum(axis=1)
    return g

def deterministic_swarm(df,m):
    A=pd.DataFrame({a:score_agent(a,df,m) for a in agents.agent})
    C=pd.DataFrame(index=df.index)
    for c in agents.coalition.unique():
        C[c]=A[agents.loc[agents.coalition==c,"agent"]].mean(axis=1)
    G=governance(df,m)
    cw={"Finance":.18,"Investment":.16,"Family":.18,"Location":.15,"Lifestyle":.10,"Risk":.13,"Governance":.10}
    out=df.copy()
    out["consensus_raw"]=sum(C[c]*w for c,w in cw.items())
    out["disagreement"]=A.std(axis=1); out["hard_pass"]=G.hard_pass; out["violations"]=G.violation_count
    out["consensus_score"]=100*np.clip(out.consensus_raw-.10*out.violations-.30*(~out.hard_pass),-1,1)
    return out,A,C,G

results,agent_scores,coalition_scores,gov=deterministic_swarm(market,mandate)
top=results.sort_values(["hard_pass","consensus_score"],ascending=[False,False]).head(15)
display(top[["property_id","neighborhood","price","beds","baths","sqft","all_in_monthly",
             "consensus_score","disagreement","violations"]])

,property_id,neighborhood,price,beds,baths,sqft,all_in_monthly,consensus_score,disagreement,violations
600,P0601,Maple Heights,738000.0,3,2.5,1581,5283.800437,57.684142,0.140432,0
1023,P1024,Westfield,576000.0,3,2.0,1625,4349.437389,56.558601,0.160281,0
877,P0878,Cedar Grove,716000.0,4,2.5,1925,5004.458835,54.194851,0.150714,0
266,P0267,Old Town,764000.0,3,2.0,1624,5580.116006,54.090595,0.154598,0
1030,P1031,Harbor Point,844000.0,3,2.5,1792,6429.535636,54.062174,0.172856,0
313,P0314,Westfield,911000.0,3,2.0,1823,6185.401819,53.540411,0.170444,0
304,P0305,Harbor Point,822000.0,3,2.0,1502,6156.492084,53.295506,0.169997,0
62,P0063,Cedar Grove,844000.0,3,2.0,1506,5825.858849,52.411770,0.135877,0
483,P0484,Maple Heights,871000.0,3,2.0,2015,6165.449386,51.472601,0.181564,0
586,P0587,Westfield,798000.0,3,2.0,1876,6067.974335,50.001221,0.152797,0


### The Hybrid Architecture: Determinism, LLMs, and the Swarm

Your observations about the `deterministic_swarm` and the feeling of a 'toolkit' are insightful and touch upon the core design philosophy of this system.

### The Role of Determinism

Indeed, the `deterministic_swarm` function (in Code Unit 5) and many other parts of this system are, as the name suggests, entirely deterministic. This is a deliberate design choice based on the principle that language models should not perform tasks that can be handled precisely and transparently by code. For instance:

*   **Financial Calculations (Code Unit 4):** Mortgage payments, taxes, insurance, and other financial metrics are calculated using explicit formulas. There is no ambiguity; the numbers are exact.
*   **Feature Engineering (Code Unit 5):** Property attributes are transformed into normalized features (e.g., affordability, school quality, safety) through defined algorithms.
*   **Individual Agent Scoring (Code Unit 5):** Each of the 50 specialists deterministically scores properties based on their assigned information sets and weighted criteria. These scores are transparent and auditable.
*   **Governance Gates (Code Unit 5):** Hard constraints (like minimum bedrooms, budget, etc.) are applied with clear, Boolean logic.
*   **Pareto Analysis (Code Unit 6):** Identifying Pareto-efficient properties is a mathematical operation, not a subjective judgment.

This deterministic foundation ensures accuracy, prevents hallucination for factual data, and maintains computational efficiency and transparency. When a precise formula or rule exists, it is always preferred over an LLM inference.

### Where the LLM Comes In

The LLM (Claude Sonnet 5 in this case) is deployed strategically where its unique capabilities—understanding nuance, synthesizing diverse information, and providing human-like reasoning—are most valuable:

1.  **Mandate Interpretation (Code Unit 3 - `interpret_buyer`):** A buyer's natural language request (e.g., "We are a family of 4 looking for at least 3 bedrooms and 2 baths. Budget around $950k.") is inherently ambiguous and unstructured. The LLM's role here is to interpret this prose and translate it into a structured, machine-readable `Buyer Mandate`. While there's a deterministic fallback, the LLM is designed to capture the subtle intentions and priorities expressed by the buyer.

2.  **Committee Deliberation (Code Unit 7 - `deliberate`):** This is perhaps the most critical LLM intervention. After the 50 deterministic agents have scored properties, governance gates have been applied, and Pareto frontiers identified, the LLM acts as a "committee secretary." It receives a distilled packet of evidence for leading candidates, which includes:
    *   Factual property attributes.
    *   Financial implications.
    *   **The diverse scores from all coalitions.**
    *   Consensus and disagreement levels among agents.
    *   Identified advocates and critics for each property.

    The LLM's task is to *deliberate* over this rich, heterogeneous (but deterministically generated) evidence. It's asked to:
    *   Identify the strongest opportunities, considering not just the highest score, but the *trade-offs*.
    *   Explain the relevant pros and cons.
    *   Surface principal concerns and potential mandate conflicts.

    This is where the LLM transcends simple data processing; it provides the nuanced, qualitative synthesis that helps a human buyer understand *why* certain properties are recommended and what their inherent trade-offs are.

### The 'Swarm' as a Toolkit, and More

The observation that the swarm may feel like a "toolkit" is inevitable. At one level, it *is* a collection of specialized tools (the 50 agents, the financial engine, the governance rules). However, the term "swarm" emphasizes the **collective intelligence** aspect derived from the **heterogeneity** and **distributed nature** of these specialists.

*   **Diversity, Not Repetition:** Instead of repeatedly querying a single LLM (which would be expensive and yield limited diversity), the system creates 50 *distinct* deterministic agents. Each agent has a specific identity (e.g., "Affordability," "Schools," "Climate Risk"), an objective function, and a focused information set. This ensures that properties are evaluated from genuinely different perspectives.
*   **Distributed Expertise:** No single agent holds all the answers. The system explicitly captures **consensus** (where agents agree) and **disagreement** (where they differ). This disagreement is treated as valuable information, not noise.
*   **Structured Deliberation:** The "swarm" isn't just about aggregating scores. It's about generating a rich tapestry of evidence (coalition scores, advocates, critics, Pareto status) that the LLM then uses for its high-level deliberation. The LLM doesn't just pick the top-scoring property; it synthesizes the *collective voice* of this diverse swarm, highlighting tensions and trade-offs.

So, while the individual components are deterministic tools, their orchestration into 50 distinct evaluators whose collective output is then deliberated upon by an LLM allows the system to achieve a form of "swarm intelligence." It moves beyond a simple weighted average to provide a more robust, auditable, and human-interpretable recommendation by simulating a diverse committee of experts.

## Code Unit 6 — Pareto efficiency, candidate packets and adversarial challenge

A weighted score is useful but dangerous. It can hide the fact that two properties represent fundamentally different trade-offs. One may dominate on family suitability and schools while another dominates on investment characteristics and walkability. Code Unit 6 therefore supplements scalar ranking with a Pareto analysis. A property is Pareto-efficient when no other candidate is at least as good across all selected dimensions and strictly better in at least one. The resulting frontier is not a single answer; it is the set of opportunities that remain defensible under different combinations of objectives.

This is particularly appropriate for real estate because there is rarely one universally dominant property. Buyers choose among bundles. A longer commute may purchase more space. A newer property may sacrifice neighborhood character. A prestigious location may reduce affordability. The Pareto frontier keeps these tensions visible.

The unit then constructs compact candidate packets for the leading properties. Each packet contains factual property attributes, financial burden, coalition scores, consensus, dissent, governance violations and Pareto status. These packets become the evidence supplied to the language-reasoning layer. This design is deliberate: Claude should deliberate over a controlled factual representation rather than receive the entire 1,200-row database and improvise its own calculations.

An explicit adversarial challenge is also created. For each leading property, the five highest-scoring specialists become its strongest advocates and the five lowest-scoring specialists become its strongest critics. This prevents consensus from becoming synonymous with conformity. A property can survive the challenge because its weaknesses are acceptable relative to the buyer mandate; alternatively, the critics may reveal a fragility that a weighted average obscures.

The unit therefore changes the nature of the recommendation. We are no longer asking, "Which row has the largest score?" We are asking, "Which opportunities remain defensible after competing objectives and explicit dissent are preserved?"

In [ ]:
# @title
# CODE UNIT 6 — Pareto frontier, candidate packets and advocate/critic challenge
def pareto_front(df,cols):
    A=df[cols].to_numpy()
    keep=np.ones(len(df),dtype=bool)
    for i in range(len(df)):
        dominated=np.all(A>=A[i],axis=1)&np.any(A>A[i],axis=1)
        if dominated.any(): keep[i]=False
    return keep

def add_pareto(results,C,m):
    out=results.copy(); out["pareto"]=False
    idx=out.index[out.hard_pass]
    if len(idx):
        P=C.loc[idx].copy()
        P["affordability"]=base_features(out.loc[idx],m)["affordability"]
        out.loc[idx,"pareto"]=pareto_front(
            P,["Finance","Investment","Family","Location","Lifestyle","Risk","affordability"]
        )
    return out

def candidate_packet(i,results,C,A):
    r=results.loc[i]
    cs=C.loc[i].sort_values(ascending=False)
    s=A.loc[i]
    return {
        "property_id":r.property_id,"neighborhood":r.neighborhood,"property_type":r.property_type,
        "style":r["style"],"price":float(r.price),"beds":int(r.beds),"baths":float(r.baths),
        "sqft":int(r.sqft),"commute_min":int(r.commute_min),"schools":round(float(r.schools),3),
        "safety":round(float(r.safety),3),"all_in_monthly":round(float(r.all_in_monthly),2),
        "days_on_market":int(r.days_on_market),"price_cut_pct":round(float(100*r.price_cut),2),
        "appreciation_est_pct":round(float(100*r.appreciation_est),2),
        "resale_liquidity":round(float(r.resale_liquidity),3),
        "consensus_score":round(float(r.consensus_score),2),
        "disagreement":round(float(r.disagreement),4),"violations":int(r.violations),
        "pareto":bool(r.pareto),
        "coalition_scores":{k:round(float(v),3) for k,v in C.loc[i].to_dict().items()},
        "strongest_advocates":s.sort_values(ascending=False).head(5).index.tolist(),
        "strongest_critics":s.sort_values().head(5).index.tolist(),
        "strongest_coalition":cs.index[0],"weakest_coalition":cs.index[-1]
    }

results=add_pareto(results,coalition_scores,mandate)
top_idx=results.sort_values(["hard_pass","consensus_score"],ascending=[False,False]).head(8).index
packets=[candidate_packet(i,results,coalition_scores,agent_scores) for i in top_idx]

challenge=pd.DataFrame([{
    "property_id":p["property_id"],
    "advocates":", ".join(p["strongest_advocates"]),
    "critics":", ".join(p["strongest_critics"]),
    "consensus":p["consensus_score"],
    "dissent":p["disagreement"],
    "pareto":p["pareto"]
} for p in packets])
display(challenge)

,property_id,advocates,critics,consensus,dissent,pareto
0,P0601,"Bathrooms, Bedrooms, Safety, Schools, Child Fr...","Style Match, Architecture, Taxes, Insurance Co...",57.68,0.1404,True
1,P1024,"Bathrooms, Bedrooms, Schools, Safety, Renovati...","Negotiation Opportunity, Rental Optionality, S...",56.56,0.1603,True
2,P0878,"Bathrooms, Bedrooms, Safety, Schools, Child Fr...","Style Match, Architecture, Commute, Taxes, Ins...",54.19,0.1507,True
3,P0267,"Bathrooms, Bedrooms, Culture, Walkability, Safety","Style Match, Architecture, Taxes, Insurance Co...",54.09,0.1546,True
4,P1031,"Bathrooms, Bedrooms, Child Friendliness, Multi...","Taxes, Insurance Cost, Affordability, Architec...",54.06,0.1729,True
5,P0314,"Safety, Bedrooms, Bathrooms, Schools, Child Fr...","Taxes, Insurance Cost, Affordability, Architec...",53.54,0.1704,True
6,P0305,"Bathrooms, Bedrooms, Safety, Resale Liquidity,...","Taxes, Insurance Cost, Style Match, Architectu...",53.30,0.1700,True
7,P0063,"Bathrooms, Bedrooms, Schools, Safety, Child Fr...","Negotiation Opportunity, Taxes, Insurance Cost...",52.41,0.1359,True




### Where the Scores Come From: Individual Agents and Coalitions

The scores for each property originate from the `deterministic_swarm` function (found in Code Unit 5), which orchestrates the evaluation by 50 individual agents.

1.  **`base_features` Function:** The first step, within the `base_features` function, is to transform raw property data (like price, bedrooms, schools, safety, commute time) into a standardized set of features. These features are normalized (typically to a 0-1 scale, where higher is better) and sometimes clipped, ensuring that all agents are evaluating properties based on comparable metrics (e.g., `affordability`, `monthly`, `schools`, `safety`).

2.  **`score_agent` Function:** This is the heart of individual agent scoring. For each of the 50 agents, the `score_agent` function does the following:
    *   **Identifies Coalition:** It first determines which `coalition` the agent belongs to (e.g., "Finance," "Family," "Investment").
    *   **Assigns Base Feature Weights (`MAPS`):** Based on its coalition, the agent is given an initial set of features to consider, with a default weight of 1.0 for each. For instance, a "Family" coalition agent would inherently prioritize features like `schools`, `safety`, `space`, `beds`, `baths`, and `lot` (as defined in the `MAPS` dictionary).
    *   **Keyword Weight Boost (`KEYWORD`):** The agent's specific `name` (e.g., "Schools," "Bedrooms," "Safety") further increases the weight of features directly related to that keyword. This ensures that an agent named "Schools" cares more about `schools` than other family-coalition agents.
    *   **Buyer Priority Adjustment:** Finally, the buyer's explicit priorities (from the `mandate`) are used to moderately adjust these weights. If the buyer states that "schools" are very important, all agents will give a slight boost to the `schools` feature, but this does not override their core expertise.
    *   **Calculates Score:** The agent's final score for a property is a weighted average of the `base_features` for that property, using the adjusted weights. This results in a single numeric score for each property from the perspective of that specific agent.

3.  **Coalition Scores:** After all 50 individual agents have scored each property, the `deterministic_swarm` function aggregates these into `coalition_scores`. For example, the **"Family" score** for a property is simply the **average of the scores of all individual agents belonging to the "Family" coalition** for that property. This provides a high-level view of how well a property satisfies the concerns of each major group of specialists.

### How the Pareto Frontier is Built

Code Unit 6 then uses these coalition scores (and an `affordability` feature derived from `base_features`) to construct the Pareto frontier.

1.  **Input for Pareto (`add_pareto` function):** The `add_pareto` function takes the `results` (which include individual property details and the derived coalition scores) and the `mandate`. It specifically focuses on properties that have passed the `hard_pass` governance gates.

2.  **Dimensions for Pareto:** The Pareto frontier is built by considering a property's performance across multiple, often conflicting, objectives. In this system, these dimensions are the **seven coalition scores (Finance, Investment, Family, Location, Lifestyle, Risk, Governance) plus an additional `affordability` feature.**

3.  **`pareto_front` Function Logic:**
    *   For each property that passed governance, the function creates a vector of its scores across these seven coalition dimensions and affordability.
    *   It then iterates through all properties to identify which ones are **Pareto-efficient**. A property is considered Pareto-efficient if no other property is *at least as good* on *all* of these dimensions and *strictly better* on *at least one* dimension. In simpler terms, you cannot improve on one aspect of a Pareto-efficient property without making another aspect worse.
    *   Properties that are *dominated* (meaning another property is better or equal on all dimensions and strictly better on at least one) are filtered out.

4.  **Result:** The `add_pareto` function updates the `results` DataFrame by adding a boolean `pareto` column, indicating `True` for properties lying on the Pareto frontier and `False` otherwise. This identifies the set of properties that represent the best trade-offs across the defined objectives, making them strong candidates for further deliberation, even if they don't have the absolute highest overall `consensus_score`.

## Code Unit 7 — Claude Sonnet 5 as coalition deliberator rather than numerical oracle

The seventh unit introduces Claude Sonnet 5 into the collective decision itself. Its role is carefully bounded. The model does not calculate mortgage payments, invent property attributes or override hard constraints. Instead, it receives the buyer mandate, the compact factual packets for the leading candidates and the deterministic coalition evidence. Claude is asked to act as a disciplined investment-committee secretary: identify the strongest opportunities, explain the relevant trade-offs, state the principal concern for each selected property and surface any unresolved conflict in the mandate.

The call requests a structured object containing a ranked shortlist, a synthesis and a list of mandate conflicts. Structured output is valuable here because the result becomes machine-readable state rather than decorative prose. The application can display the synthesis while retaining property identifiers and concern fields for later workflows.

Only one deliberation call is required for the shortlist. This is an important architectural compromise. The system still contains 50 agents because their distinct scores and roles determine the evidence. Claude then reasons over that heterogeneous evidence at committee level. In a more expensive research configuration, one could make separate calls for each coalition or even each agent. The present notebook favors a design that can be run repeatedly in Colab without turning a simple property search into hundreds of API calls.

If the live call fails, the notebook constructs a deterministic deliberation from the leading candidates and records the failure in `LLM_LOG`. The user therefore always receives an answer, but the provenance tells us whether the language synthesis came from Sonnet 5 or from fallback logic.

This hybrid pattern—probabilistic interpretation over deterministic evidence—is central to the notebook. It assigns language models to ambiguity and synthesis while leaving arithmetic and enforcement to code.

In [ ]:
# @title
# CODE UNIT 7 — Claude Sonnet 5 committee deliberation over deterministic evidence
DELIB_SCHEMA={
    "type":"object",
    "properties":{
        "shortlist":{
            "type":"array","minItems":1,"maxItems":5,
            "items":{
                "type":"object",
                "properties":{
                    "property_id":{"type":"string"},
                    "case_for":{"type":"string"},
                    "principal_concern":{"type":"string"},
                    "fit_summary":{"type":"string"}
                },
                "required":["property_id","case_for","principal_concern","fit_summary"],
                "additionalProperties":False
            }
        },
        "synthesis":{"type":"string"},
        "mandate_conflicts":{"type":"array","items":{"type":"string"}}
    },
    "required":["shortlist","synthesis","mandate_conflicts"],
    "additionalProperties":False
}

def fallback_deliberation(packets):
    chosen=packets[:3]
    return {
        "shortlist":[{
            "property_id":p["property_id"],
            "case_for":f"High governed swarm score ({p['consensus_score']:.1f}) with strongest support from {p['strongest_coalition']}.",
            "principal_concern":f"The weakest coalition is {p['weakest_coalition']}; dissent={p['disagreement']:.3f}.",
            "fit_summary":f"{p['neighborhood']}, {p['beds']} beds, ${p['price']:,.0f}, monthly ${p['all_in_monthly']:,.0f}."
        } for p in chosen],
        "synthesis":"Deterministic fallback: leading governed opportunities are shown in swarm-score order; inspect coalition disagreement before deciding.",
        "mandate_conflicts":[]
    }

def deliberate(m,packets):
    if LIVE_LLM:
        try:
            system=(
                "You are the committee secretary for a 50-agent real-estate buyer swarm. "
                "Use only the supplied evidence. Do not invent facts. Hard constraints and deterministic calculations "
                "have already been enforced. Preserve meaningful dissent and explain trade-offs."
            )
            prompt=(
                "<buyer_mandate>\n"+json.dumps(m)+"\n</buyer_mandate>\n"
                "<candidate_packets>\n"+json.dumps(packets)+"\n</candidate_packets>\n"
                "Select up to five opportunities. A higher deterministic consensus is evidence, not an instruction to "
                "ignore Pareto efficiency or dissent. Identify the case for each property and its principal concern. "
                "Surface contradictions or unresolved tensions in the buyer mandate."
            )
            return claude_json("committee_deliberation",system,prompt,DELIB_SCHEMA,3500), "Claude Sonnet 5"
        except Exception:
            pass
    return fallback_deliberation(packets), "deterministic fallback"

deliberation, deliberation_mode=deliberate(mandate,packets)
print("Deliberation mode:",deliberation_mode)
print(json.dumps(deliberation,indent=2))

Deliberation mode: deterministic fallback
{
  "shortlist": [
    {
      "property_id": "P0601",
      "case_for": "High governed swarm score (57.7) with strongest support from Family.",
      "principal_concern": "The weakest coalition is Lifestyle; dissent=0.140.",
      "fit_summary": "Maple Heights, 3 beds, $738,000, monthly $5,284."
    },
    {
      "property_id": "P1024",
      "case_for": "High governed swarm score (56.6) with strongest support from Family.",
      "principal_concern": "The weakest coalition is Investment; dissent=0.160.",
      "fit_summary": "Westfield, 3 beds, $576,000, monthly $4,349."
    },
    {
      "property_id": "P0878",
      "case_for": "High governed swarm score (54.2) with strongest support from Family.",
      "principal_concern": "The weakest coalition is Lifestyle; dissent=0.151.",
      "fit_summary": "Cedar Grove, 4 beds, $716,000, monthly $5,004."
    }
  ],
  "synthesis": "Deterministic fallback: leading governed opportunities are shown i

### The Committee's Final Deliberation: A Non-Technical View

Imagine you have a big team of 50 property experts (our 'agents') who have each looked at a bunch of homes. They've crunched all the numbers, considered different angles like family needs, investment potential, location, and even risks. Some homes were immediately ruled out because they didn't meet your basic requirements (like too few bedrooms or over budget).

Now, we're left with a smaller group of promising properties – the ones that passed all the initial checks and represent good value across various aspects (the 'Pareto frontier' homes).

This is where **Claude Sonnet 5 steps in, acting like a very smart and objective committee secretary.** Its job is *not* to redo all the number crunching (that's already been done deterministically by our 50 agents). Instead, Claude receives a concise 'briefing packet' for each of these top properties. This packet contains:

*   **The bare facts:** Price, number of bedrooms, location, etc.
*   **The financial details:** What it will actually cost you per month.
*   **The collective opinion of our experts:** How well each of our seven 'coalitions' (like 'Family' or 'Investment') scored the property.
*   **Who loved it and who didn't:** It even highlights which few agents were the strongest champions and which were the strongest critics for each home, and how much overall disagreement there was.
*   **Its Pareto status:** Whether it represents a best-in-class trade-off.


With all this distilled information, Claude's role is to:

1.  **Synthesize:** Read through all the briefing packets and your original wishes (the 'buyer mandate').
2.  **Shortlist:** Pick out the very best few properties that seem to align most closely with what you're looking for, considering all the evidence.
3.  **Explain the 'Why':** For each shortlisted property, Claude will explain *why* it's a good choice (its 'case for'). This isn't just about the highest score; it's about the overall balance and what our experts liked about it.
4.  **Identify Trade-offs:** Crucially, Claude will also point out the 'principal concern' for each property. No home is perfect, and Claude helps you understand what compromises or potential downsides you might encounter. This is where it highlights the tensions and disagreements among our 50 experts.
5.  **Spot Conflicts:** If your original request had any hidden contradictions (e.g., wanting a huge backyard *and* a tiny commute in a dense city), Claude might flag these as 'mandate conflicts,' helping you clarify your own priorities.


In essence, Claude transforms a mountain of data and a chorus of expert opinions into a clear, reasoned recommendation, much like a trusted advisor would, without inventing facts or doing calculations itself. It's the human-like intelligence that makes sense of all the precise, deterministic work that came before it.

## Code Unit 8 — Robustness: asking whether the recommendation survives changing interest rates

Real-estate decisions are unusually sensitive to financing conditions. A recommendation that looks compelling at one mortgage rate can become infeasible when rates rise, while a less expensive property may move to the front of the opportunity set. The eighth unit therefore performs an explicit interest-rate sensitivity analysis rather than presenting the baseline recommendation as if financing were fixed.

The function reruns the financial engine and the complete deterministic swarm across several mortgage-rate scenarios. For each rate, monthly carrying costs are recomputed, governance conditions are reevaluated, all 50 specialist scores are regenerated, coalition scores are rebuilt and the leading feasible property is identified. The result shows whether the same opportunity remains attractive across regimes or whether the recommendation is fragile.

This is more than a standard sensitivity table. It demonstrates an important principle for autonomous decision systems: the answer should be understood as a function of assumptions. A system that reports one optimal property without exposing the assumptions that support it creates false precision. Here, the mortgage rate is only the first scenario dimension. A production application could repeat the same procedure for down payment, holding horizon, expected appreciation, renovation costs, insurance shocks, commute changes, school priorities or family-size scenarios.

The analysis also creates a natural role for the Risk coalition. Robustness is not simply another weighted feature. It is evidence about how the entire recommendation behaves when the environment changes. A property that remains near the top across plausible financing conditions may deserve a different level of confidence from a property whose ranking collapses after a small rate increase.

The output is intentionally simple: rate, top property, price, monthly cost and swarm score. Its purpose is to make model dependence visible before the app packages the recommendation into a polished conversational experience.

In [ ]:
# @title
# CODE UNIT 8 — Interest-rate sensitivity and recommendation robustness
def rate_sensitivity(m,rates=(.045,.055,.065,.075,.085)):
    out=[]
    for rate in rates:
        mm=m.copy(); mm["mortgage_rate"]=rate
        z=enrich_financials(DATA.search(mm),mm)
        R,A,C,G=deterministic_swarm(z,mm)
        feasible=R[R.hard_pass].sort_values("consensus_score",ascending=False)
        if len(feasible):
            i=feasible.index[0]
            out.append({
                "mortgage_rate":rate,
                "top_property":R.loc[i,"property_id"],
                "price":R.loc[i,"price"],
                "monthly_cost":R.loc[i,"all_in_monthly"],
                "score":R.loc[i,"consensus_score"]
            })
    return pd.DataFrame(out)

sensitivity=rate_sensitivity(mandate)
display(sensitivity.style.format({
    "mortgage_rate":"{:.2%}","price":"${:,.0f}","monthly_cost":"${:,.0f}","score":"{:.1f}"
}))

fig_rate=px.line(
    sensitivity,x="mortgage_rate",y="monthly_cost",markers=True,
    hover_data=["top_property","score"],
    title="Financing robustness: monthly cost of the leading opportunity by mortgage rate"
)
fig_rate.update_xaxes(tickformat=".1%")
#fig_rate.show()

,mortgage_rate,top_property,price,monthly_cost,score
0,4.50%,P0601,"$738,000","$4,544",58.8
1,5.50%,P0601,"$738,000","$4,904",58.3
2,6.50%,P0601,"$738,000","$5,284",57.7
3,7.50%,P0601,"$738,000","$5,680",57.1
4,8.50%,P0601,"$738,000","$6,092",56.5


## Code Unit 9 — Integrating the complete swarm into one callable decision pipeline

By this stage, the notebook contains all of the components of the architecture, but an application needs a single callable function. Code Unit 9 assembles the full sequence into `run_swarm`. This function is the operational heart of the prototype. It accepts an ordinary-language buyer conversation and returns a narrative recommendation, a tabular shortlist, a coalition visualization, the inferred mandate and a provenance summary.

The function first interprets the conversation. If Claude Sonnet 5 is available, the live structured mandate parser is used; otherwise the deterministic parser supplies the mandate. The synthetic data adapter then retrieves the property universe and the financial engine recomputes all carrying costs using the mandate's financing assumptions. Fifty specialist scores are generated, coalition scores are aggregated, and the governance gate identifies constraint violations.

The function ranks the market but does not immediately present the winner. It identifies a compact leading set, computes the Pareto frontier for feasible candidates, and creates the evidence packets used for deliberation. If live language reasoning is enabled, Sonnet 5 synthesizes the shortlist and its trade-offs. If not, deterministic synthesis is used. The table shown to the user retains factual columns such as price, bedrooms, area, commute, schools, safety, monthly cost, appreciation estimate, consensus, disagreement and violations.

The Plotly visualization exposes coalition differences across the leading properties. This is pedagogically important because it lets the user see that the recommendation is not generated by a single opaque number. Finance, Family, Investment, Location, Lifestyle, Risk and Governance may value the same property differently.

Finally, the function reports provenance: model identifier, live/fallback mode and the recent language-model log. The recommendation is therefore not only explainable in substantive terms; it is also operationally auditable.

In [ ]:
# @title
# CODE UNIT 9 — Complete end-to-end swarm pipeline
def run_swarm(interview,n_show=8):
    m, mandate_source=interpret_buyer(interview)
    df=enrich_financials(DATA.search(m),m)
    R,A,C,G=deterministic_swarm(df,m)
    R=add_pareto(R,C,m)

    ranked=R.sort_values(["hard_pass","consensus_score"],ascending=[False,False]).head(int(n_show)).copy()
    candidate_indices=ranked.index[:min(8,len(ranked))]
    local_packets=[candidate_packet(i,R,C,A) for i in candidate_indices]
    committee, committee_source=deliberate(m,local_packets)

    show=ranked[["property_id","neighborhood","property_type","style","price","beds","baths","sqft",
                 "commute_min","schools","safety","all_in_monthly","days_on_market","appreciation_est",
                 "resale_liquidity","consensus_score","disagreement","violations","pareto"]].copy()
    show["price"]=show.price.map(lambda x:f"${x:,.0f}")
    show["all_in_monthly"]=show.all_in_monthly.map(lambda x:f"${x:,.0f}")
    show["appreciation_est"]=show.appreciation_est.map(lambda x:f"{100*x:.1f}%")
    show["consensus_score"]=show.consensus_score.round(1)
    show["disagreement"]=show.disagreement.round(3)

    shortlist_lines=[]
    for j,item in enumerate(committee["shortlist"],1):
        shortlist_lines.append(
            f"**{j}. {item['property_id']}** — {item['fit_summary']}  \n"
            f"*Case:* {item['case_for']}  \n"
            f"*Concern:* {item['principal_concern']}"
        )

    narrative=(
        "### 50-agent committee synthesis\n\n"
        + committee["synthesis"] + "\n\n"
        + "\n\n".join(shortlist_lines)
        + ("\n\n**Unresolved mandate tensions:** " + "; ".join(committee["mandate_conflicts"])
           if committee["mandate_conflicts"] else "")
        + f"\n\n**Execution:** mandate={mandate_source}; committee={committee_source}; model={MODEL}."
    )

    chart=C.loc[ranked.index[:5]].copy()
    chart["property_id"]=ranked.property_id.values[:len(chart)]
    long=chart.melt(id_vars="property_id",var_name="Coalition",value_name="Score")
    fig=px.bar(long,x="property_id",y="Score",color="Coalition",barmode="group",
               title="How the seven coalitions view the leading opportunities")

    provenance={
        "model":MODEL,"live_llm":LIVE_LLM,
        "mandate_source":mandate_source,"committee_source":committee_source,
        "recent_llm_log":LLM_LOG[-6:]
    }
    return narrative,show,fig,json.dumps(m,indent=2),json.dumps(provenance,indent=2)

_smoke=run_swarm(sample_interview,5)
print("Pipeline ready. Returned shortlist rows:",len(_smoke[1]))

Pipeline ready. Returned shortlist rows: 5


## Code Unit 10 — A conversational Colab application and the bridge to a production product

The final code unit turns the research architecture into something a client can actually touch. Gradio is used because it runs naturally inside Google Colab and requires very little infrastructure. The interface intentionally begins with a conversation box rather than a wall of filters. This preserves the central idea of the project: the client should describe the problem in human terms, while the system is responsible for translating that language into a formal mandate.

The default conversation provides an example involving a family, budget, bedroom and bathroom requirements, schools, safety, commute, architectural preference, holding horizon and renovation tolerance. The user can replace it with any narrative. Pressing the button launches the complete pipeline from Code Unit 9. The interface then displays the swarm synthesis, a structured shortlist, a coalition comparison chart, the inferred Buyer Mandate and model provenance.

This is deliberately a basic application rather than a polished consumer product. Its purpose is to demonstrate that the architecture closes the loop from conversation to opportunity. Nevertheless, the same interface can evolve substantially. A production version could add a map, property photographs, saved homes, rejection reasons, comparable sales, mortgage scenarios, inspection documents, school evidence and neighborhood reports. Most importantly, it could preserve memory across interactions.

That memory would transform the application. Suppose the buyer repeatedly rejects suburban properties despite initially assigning schools the highest priority. The system should update its belief about the client's latent preferences. Saved, rejected, toured and revisited properties become revealed-preference observations. The next search would therefore begin not merely from what the client originally said but from what the client has demonstrated through behavior.

The Gradio app is consequently the endpoint of this notebook but the beginning of the larger architecture: an adaptive artificial buyer's committee that learns the client while keeping its calculations, disagreements, evidence and governance visible.

In [ ]:
# @title
# CODE UNIT 10 — Premium conversational real-estate swarm app
# 50 SHADES OF AI | Claude Sonnet 5 | 50-Agent Decision Architecture

import gradio as gr

# ---------------------------------------------------------------------
# PREMIUM UI THEME
# ---------------------------------------------------------------------

APP_CSS = """
:root {
    --navy: #071f3d;
    --navy2: #0b315c;
    --blue: #1769ff;
    --blue2: #4a8cff;
    --cyan: #18b6d9;
    --green: #16a36a;
    --gold: #d89b31;
    --purple: #7557e8;
    --ink: #10233f;
    --muted: #64748b;
    --line: #dce6f2;
    --panel: rgba(255,255,255,.96);
    --soft: #f5f8fc;
}

/* -------------------------------------------------------
   PAGE
------------------------------------------------------- */

.gradio-container {
    max-width: 1480px !important;
    margin: 0 auto !important;
    background:
        radial-gradient(circle at 5% 0%, rgba(23,105,255,.08), transparent 24rem),
        radial-gradient(circle at 95% 8%, rgba(117,87,232,.07), transparent 28rem),
        #f4f7fb !important;
    color: var(--ink);
    font-family:
        Inter, ui-sans-serif, system-ui, -apple-system,
        BlinkMacSystemFont, "Segoe UI", sans-serif !important;
}

footer {
    display: none !important;
}

/* -------------------------------------------------------
   HERO
------------------------------------------------------- */

.hero {
    position: relative;
    overflow: hidden;
    padding: 38px 42px;
    border-radius: 28px;
    margin: 8px 0 22px 0;

    background:
        radial-gradient(circle at 85% 10%, rgba(50,135,255,.28), transparent 24rem),
        linear-gradient(120deg, #061a34 0%, #092d57 58%, #0a4073 100%);

    box-shadow: 0 22px 60px rgba(6,30,60,.20);
}

.hero::after {
    content: "";
    position: absolute;
    width: 420px;
    height: 420px;
    right: -150px;
    bottom: -260px;
    border: 1px solid rgba(255,255,255,.15);
    border-radius: 50%;
}

.eyebrow {
    color: #80b9ff;
    font-size: 12px;
    font-weight: 800;
    letter-spacing: .16em;
    text-transform: uppercase;
    margin-bottom: 12px;
}

.hero h1 {
    color: white !important;
    font-family: Georgia, "Times New Roman", serif;
    font-size: 46px !important;
    line-height: 1.02 !important;
    margin: 0 !important;
    letter-spacing: -.025em;
}

.hero-subtitle {
    color: #dbeafe;
    font-size: 17px;
    line-height: 1.55;
    max-width: 850px;
    margin-top: 14px;
}

.hero-badges {
    display: flex;
    gap: 9px;
    flex-wrap: wrap;
    margin-top: 24px;
}

.hero-badge {
    color: white;
    font-size: 12px;
    font-weight: 700;
    padding: 7px 12px;
    border: 1px solid rgba(255,255,255,.18);
    border-radius: 999px;
    background: rgba(255,255,255,.08);
    backdrop-filter: blur(8px);
}

/* -------------------------------------------------------
   SECTION TITLES
------------------------------------------------------- */

.section-kicker {
    font-size: 11px;
    text-transform: uppercase;
    letter-spacing: .15em;
    font-weight: 800;
    color: var(--blue);
    margin-bottom: 5px;
}

.section-title {
    font-family: Georgia, "Times New Roman", serif;
    color: var(--navy);
    font-size: 27px;
    font-weight: 700;
    margin-bottom: 4px;
}

.section-copy {
    color: var(--muted);
    line-height: 1.55;
    font-size: 14px;
    margin-bottom: 12px;
}

/* -------------------------------------------------------
   CARDS
------------------------------------------------------- */

.premium-card {
    background: var(--panel);
    border: 1px solid rgba(202,216,232,.85);
    border-radius: 22px;
    padding: 20px !important;
    box-shadow: 0 8px 28px rgba(20,49,82,.07);
}

.premium-card:hover {
    box-shadow: 0 12px 36px rgba(20,49,82,.10);
}

.input-card {
    border-top: 4px solid var(--blue);
}

.results-card {
    border-top: 4px solid var(--green);
}

/* -------------------------------------------------------
   TEXT INPUT
------------------------------------------------------- */

#buyer-interview textarea {
    font-size: 15px !important;
    line-height: 1.58 !important;
    min-height: 240px !important;
    border-radius: 16px !important;
}

/* -------------------------------------------------------
   PRIMARY BUTTON
------------------------------------------------------- */

#swarm-button {
    border: none !important;
    border-radius: 14px !important;
    min-height: 52px !important;

    background:
        linear-gradient(100deg, #0c56dc 0%, #1877ff 55%, #3291ff 100%) !important;

    color: white !important;
    font-weight: 800 !important;
    font-size: 15px !important;
    letter-spacing: .01em;

    box-shadow: 0 10px 24px rgba(23,105,255,.26) !important;
}

#swarm-button:hover {
    transform: translateY(-1px);
    box-shadow: 0 14px 30px rgba(23,105,255,.34) !important;
}

/* -------------------------------------------------------
   PROCESS STRIP
------------------------------------------------------- */

.process-strip {
    display: grid;
    grid-template-columns: repeat(5, 1fr);
    gap: 10px;
    margin: 6px 0 24px 0;
}

.process-item {
    background: white;
    border: 1px solid var(--line);
    border-radius: 16px;
    padding: 13px 14px;
}

.process-number {
    display: inline-flex;
    width: 27px;
    height: 27px;
    align-items: center;
    justify-content: center;
    border-radius: 50%;
    color: white;
    background: var(--blue);
    font-size: 12px;
    font-weight: 800;
    margin-bottom: 7px;
}

.process-label {
    display: block;
    font-size: 12px;
    font-weight: 800;
    color: var(--navy);
}

.process-note {
    display: block;
    font-size: 11px;
    color: var(--muted);
    margin-top: 2px;
}

/* -------------------------------------------------------
   METRIC CARDS
------------------------------------------------------- */

.metric-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 10px;
}

.metric {
    border-radius: 15px;
    padding: 14px;
    background: linear-gradient(180deg,#f9fbfe,#f2f6fb);
    border: 1px solid var(--line);
}

.metric-label {
    color: var(--muted);
    font-size: 10px;
    text-transform: uppercase;
    letter-spacing: .08em;
    font-weight: 800;
}

.metric-value {
    color: var(--navy);
    font-size: 19px;
    font-weight: 800;
    margin-top: 3px;
}

/* -------------------------------------------------------
   OUTPUT
------------------------------------------------------- */

#committee-output {
    background: linear-gradient(135deg,#ffffff,#f8fbff);
    border-left: 5px solid var(--blue);
    border-radius: 18px;
    padding: 18px 22px;
}

#committee-output h3 {
    color: var(--navy) !important;
    font-family: Georgia, "Times New Roman", serif;
    font-size: 24px !important;
}

/* -------------------------------------------------------
   DATAFRAME
------------------------------------------------------- */

#shortlist table {
    font-size: 12px !important;
}

#shortlist {
    border-radius: 16px;
    overflow: hidden;
}

/* -------------------------------------------------------
   ACCORDION / AUDIT
------------------------------------------------------- */

.audit-note {
    color: var(--muted);
    font-size: 12px;
    line-height: 1.5;
}

/* -------------------------------------------------------
   STATUS
------------------------------------------------------- */

.status-live {
    display: inline-flex;
    align-items: center;
    gap: 7px;
    padding: 7px 11px;
    border-radius: 999px;
    background: #eaf8f1;
    color: #08784b;
    font-weight: 800;
    font-size: 11px;
}

.status-dot {
    width: 7px;
    height: 7px;
    border-radius: 50%;
    background: #14a36b;
}

/* -------------------------------------------------------
   RESPONSIVE
------------------------------------------------------- */

@media (max-width: 900px) {

    .hero {
        padding: 28px 24px;
    }

    .hero h1 {
        font-size: 34px !important;
    }

    .process-strip {
        grid-template-columns: 1fr;
    }

    .metric-grid {
        grid-template-columns: repeat(2, 1fr);
    }
}
"""


# ---------------------------------------------------------------------
# HTML COMPONENTS
# ---------------------------------------------------------------------

HERO = """
<div class="hero">
    <div class="eyebrow">50 SHADES OF AI · DECISION INTELLIGENCE</div>

    <h1>Real Estate Opportunity Swarm</h1>

    <div class="hero-subtitle">
        A conversational buyer advisory system in which
        <strong>50 specialized AI agents</strong> examine affordability,
        investment value, family fit, location, lifestyle and risk —
        while deterministic governance keeps the decision anchored
        to the buyer's mandate.
    </div>

    <div class="hero-badges">
        <span class="hero-badge">Claude Sonnet 5</span>
        <span class="hero-badge">50 Specialized Agents</span>
        <span class="hero-badge">7 Expert Coalitions</span>
        <span class="hero-badge">Pareto Analysis</span>
        <span class="hero-badge">Governed Decisions</span>
    </div>
</div>
"""

PROCESS = """
<div class="process-strip">

    <div class="process-item">
        <span class="process-number">1</span>
        <span class="process-label">CONVERSE</span>
        <span class="process-note">Describe your life and objectives</span>
    </div>

    <div class="process-item">
        <span class="process-number">2</span>
        <span class="process-label">FORMALIZE</span>
        <span class="process-note">Build the Buyer Mandate</span>
    </div>

    <div class="process-item">
        <span class="process-number">3</span>
        <span class="process-label">DELIBERATE</span>
        <span class="process-note">50 specialists challenge the market</span>
    </div>

    <div class="process-item">
        <span class="process-number">4</span>
        <span class="process-label">GOVERN</span>
        <span class="process-note">Constraints cannot be voted away</span>
    </div>

    <div class="process-item">
        <span class="process-number">5</span>
        <span class="process-label">EXPLAIN</span>
        <span class="process-note">Receive a defensible shortlist</span>
    </div>

</div>
"""

METRICS = """
<div class="metric-grid">

    <div class="metric">
        <div class="metric-label">Specialists</div>
        <div class="metric-value">50 agents</div>
    </div>

    <div class="metric">
        <div class="metric-label">Perspectives</div>
        <div class="metric-value">7 coalitions</div>
    </div>

    <div class="metric">
        <div class="metric-label">Decision Logic</div>
        <div class="metric-value">Governed</div>
    </div>

    <div class="metric">
        <div class="metric-label">Reasoning Layer</div>
        <div class="metric-value">Sonnet 5</div>
    </div>

</div>
"""


# ---------------------------------------------------------------------
# GRADIO APPLICATION
# ---------------------------------------------------------------------

with gr.Blocks(
    title="50 Shades of AI — Real Estate Opportunity Swarm",
    css=APP_CSS,
    fill_width=True
) as demo:

    # -------------------------------------------------------------
    # HERO
    # -------------------------------------------------------------

    gr.HTML(HERO)

    gr.HTML(PROCESS)

    # -------------------------------------------------------------
    # WORKSPACE
    # -------------------------------------------------------------

    with gr.Row(equal_height=False):

        # LEFT — BUYER CONVERSATION
        with gr.Column(scale=5, elem_classes=["premium-card", "input-card"]):

            gr.HTML("""
                <div class="section-kicker">Buyer Brief</div>
                <div class="section-title">Tell us what matters.</div>
                <div class="section-copy">
                    Speak naturally. Describe your family, budget, preferred
                    neighborhoods, commute, schools, style, investment horizon,
                    renovation tolerance and anything else that should influence
                    the decision.
                </div>
            """)

            interview = gr.Textbox(
                value=sample_interview,
                lines=11,
                label="Your conversation",
                placeholder=(
                    "For example: We are a family of four. "
                    "Schools matter enormously, but we also want a reasonable "
                    "commute and a home with strong resale potential..."
                ),
                elem_id="buyer-interview"
            )

            nshow = gr.Slider(
                minimum=3,
                maximum=15,
                value=8,
                step=1,
                label="Opportunity set",
                info="Choose how many leading properties you want the swarm to surface."
            )

            btn = gr.Button(
                "RUN THE 50-AGENT DECISION SWARM",
                variant="primary",
                elem_id="swarm-button"
            )

            gr.HTML("""
                <div style="
                    margin-top:10px;
                    padding:12px 14px;
                    background:#f5f8fc;
                    border:1px solid #dce6f2;
                    border-radius:14px;
                    font-size:12px;
                    color:#64748b;
                    line-height:1.5;
                ">
                    The swarm does not simply search for the highest-scoring
                    property. It preserves disagreement, tests hard constraints,
                    identifies Pareto-efficient alternatives and asks the AI
                    committee to explain the trade-offs.
                </div>
            """)

        # RIGHT — SYSTEM OVERVIEW
        with gr.Column(scale=4, elem_classes=["premium-card"]):

            gr.HTML("""
                <div class="section-kicker">Decision Architecture</div>
                <div class="section-title">Your digital investment committee.</div>
                <div class="section-copy">
                    Each opportunity is examined from multiple institutional
                    perspectives before it reaches the final shortlist.
                </div>
            """)

            gr.HTML(METRICS)

            gr.HTML("""
                <div style="
                    margin-top:16px;
                    padding:17px;
                    border-radius:16px;
                    background:linear-gradient(135deg,#071f3d,#0b3c6f);
                    color:white;
                ">
                    <div style="
                        font-size:11px;
                        text-transform:uppercase;
                        letter-spacing:.1em;
                        color:#82b9ff;
                        font-weight:800;
                    ">
                        Seven Expert Coalitions
                    </div>

                    <div style="
                        margin-top:10px;
                        line-height:1.9;
                        font-size:13px;
                    ">
                        <b>Finance</b> · affordability & carrying cost<br>
                        <b>Investment</b> · resale, appreciation & optionality<br>
                        <b>Family</b> · schools, safety & usable space<br>
                        <b>Location</b> · commute, walkability & amenities<br>
                        <b>Lifestyle</b> · architecture, condition & fit<br>
                        <b>Risk</b> · climate, age & downside exposure<br>
                        <b>Governance</b> · constraints, evidence & discipline
                    </div>
                </div>
            """)

            gr.HTML("""
                <div style="margin-top:16px;">
                    <span class="status-live">
                        <span class="status-dot"></span>
                        CLAUDE SONNET 5 READY WHEN API SECRET IS AVAILABLE
                    </span>
                </div>
            """)

    # -------------------------------------------------------------
    # RESULTS HEADER
    # -------------------------------------------------------------

    gr.HTML("""
        <div style="margin-top:30px;margin-bottom:10px;">
            <div class="section-kicker">Decision Room</div>
            <div class="section-title">Swarm Intelligence Report</div>
            <div class="section-copy">
                The output combines the committee's narrative judgment with
                the underlying governed opportunity set and coalition evidence.
            </div>
        </div>
    """)

    # -------------------------------------------------------------
    # COMMITTEE SYNTHESIS
    # -------------------------------------------------------------

    with gr.Column(elem_classes=["premium-card", "results-card"]):

        gr.HTML("""
            <div style="
                display:flex;
                justify-content:space-between;
                align-items:center;
                margin-bottom:10px;
            ">
                <div>
                    <div class="section-kicker">Executive View</div>
                    <div class="section-title">Committee Recommendation</div>
                </div>

                <div style="
                    padding:7px 11px;
                    border-radius:999px;
                    background:#eef4ff;
                    color:#1769ff;
                    font-size:11px;
                    font-weight:800;
                ">
                    EXPLAINABLE AI
                </div>
            </div>
        """)

        narrative = gr.Markdown(
            value=(
                "### Awaiting the buyer brief\n\n"
                "Run the swarm to generate the committee's synthesis, "
                "leading opportunities, principal concerns and unresolved "
                "trade-offs."
            ),
            elem_id="committee-output"
        )

    # -------------------------------------------------------------
    # OPPORTUNITY SET + COALITION VIEW
    # -------------------------------------------------------------

    with gr.Row(equal_height=False):

        with gr.Column(scale=6, elem_classes=["premium-card"]):

            gr.HTML("""
                <div class="section-kicker">Opportunity Set</div>
                <div class="section-title">Governed Shortlist</div>
                <div class="section-copy">
                    Leading properties after mandate constraints,
                    50-agent scoring, coalition aggregation and
                    Pareto analysis.
                </div>
            """)

            table = gr.Dataframe(
                label="",
                interactive=False,
                wrap=True,
                elem_id="shortlist"
            )

        with gr.Column(scale=5, elem_classes=["premium-card"]):

            gr.HTML("""
                <div class="section-kicker">Inside the Swarm</div>
                <div class="section-title">Coalition Perspectives</div>
                <div class="section-copy">
                    See where Finance, Investment, Family, Location,
                    Lifestyle, Risk and Governance agree — and where
                    they do not.
                </div>
            """)

            plot = gr.Plot(label="")

    # -------------------------------------------------------------
    # TRANSPARENCY / AUDIT LAYER
    # -------------------------------------------------------------

    gr.HTML("""
        <div style="margin-top:28px;margin-bottom:8px;">
            <div class="section-kicker">Transparency Layer</div>
            <div class="section-title">Inspect the machine.</div>
            <div class="section-copy">
                The client experience remains simple, while the complete
                mandate and execution provenance remain available for
                audit, teaching and research.
            </div>
        </div>
    """)

    with gr.Accordion(
        "Buyer Mandate — structured interpretation of the conversation",
        open=False
    ):
        mandate_json = gr.Code(
            label="Buyer Mandate",
            language="json",
            lines=18
        )

    with gr.Accordion(
        "Model Provenance — how this recommendation was produced",
        open=False
    ):
        provenance_json = gr.Code(
            label="Execution Provenance",
            language="json",
            lines=16
        )

    # -------------------------------------------------------------
    # FOOTER
    # -------------------------------------------------------------

    gr.HTML("""
        <div style="
            margin:28px 0 12px 0;
            padding:20px 24px;
            border-radius:18px;
            background:#071f3d;
            color:#dbeafe;
            display:flex;
            justify-content:space-between;
            gap:20px;
            flex-wrap:wrap;
            align-items:center;
        ">
            <div>
                <div style="
                    color:white;
                    font-family:Georgia,serif;
                    font-size:19px;
                    font-weight:700;
                ">
                    50 SHADES OF AI
                </div>

                <div style="
                    margin-top:4px;
                    font-size:12px;
                    color:#9fc5f5;
                ">
                    From conversation to governed opportunity.
                </div>
            </div>

            <div style="
                font-size:11px;
                color:#9fb3ca;
                max-width:620px;
                line-height:1.5;
                text-align:right;
            ">
                Research prototype. Outputs are decision-support information,
                not financial, legal, tax, appraisal, brokerage or investment advice.
            </div>
        </div>
    """)

    # -------------------------------------------------------------
    # EXECUTION
    # -------------------------------------------------------------

    btn.click(
        fn=run_swarm,
        inputs=[interview, nshow],
        outputs=[
            narrative,
            table,
            plot,
            mandate_json,
            provenance_json
        ],
        show_progress="full"
    )


# ---------------------------------------------------------------------
# LAUNCH
# ---------------------------------------------------------------------

demo.launch(
    share=True,
    debug=False,
    show_error=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dd3260bc7f3f5b618f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Conclusion

This notebook began from a deceptively familiar problem: finding a property to buy. What appears on conventional portals as a filtering exercise is, on closer inspection, a complex decision under uncertainty. Price, bedrooms and location are only the first layer. Financing conditions determine affordability; taxes, insurance, maintenance and homeowners' association charges determine carrying cost; schools, safety, commute and amenities determine family and location utility; age and condition determine future expenditure; architectural character influences subjective fit; and resale liquidity, appreciation and rental optionality connect the home to the buyer's future balance sheet. None of these dimensions exists independently, and no universal weighting system can tell every buyer what matters most.

The 50-agent architecture developed here addresses this complexity by refusing to collapse the problem too early. Instead of treating intelligence as one model producing one answer, the notebook creates a heterogeneous society of specialists. Finance agents, investment agents, family agents, location agents, lifestyle agents, risk agents and governance agents observe the same opportunity through different lenses. Their differences are not decorative personas. They are encoded in the features they emphasize and the objectives they represent. The system therefore preserves a property of real institutional decision making: reasonable experts can examine the same facts and disagree because they are responsible for different dimensions of the problem.

The resulting swarm is hybrid rather than purely generative. Deterministic code performs the tasks for which deterministic code is superior. Mortgage payments are calculated by formula. Taxes, insurance proxies and maintenance estimates are added explicitly. Hard constraints are evaluated transparently. Scores and sensitivity analyses can be reproduced. Pareto efficiency is computed rather than narrated. This is essential because an autonomous financial or real-estate system should not ask a language model to improvise arithmetic that can be verified exactly.

Claude Sonnet 5 is used where language intelligence creates genuine value. It interprets the buyer's natural-language conversation and converts it into a structured mandate. It then deliberates over compact factual packets created by the deterministic system, identifying trade-offs, concerns and unresolved conflicts. The architecture therefore treats the language model as a semantic and deliberative layer rather than as an omniscient database or numerical oracle. If the live API is unavailable, deterministic fallbacks preserve the notebook's functionality, and the provenance log records what happened.

Governance is equally important. The swarm cannot simply vote away a hard requirement because an attractive property excites the lifestyle coalition. Constraints remain constraints. At the same time, the system does not pretend that all requirements are equally rigid. Commute, monthly burden, style and renovation tolerance can be represented as softer preferences whose importance changes with the buyer. This separation between mandate, evidence, deliberation and governance is one of the most transferable ideas in the notebook.

The Pareto analysis further prevents false precision. Real-estate opportunities are bundles of advantages and sacrifices. A property may offer superior schools and space but a weaker commute. Another may deliver walkability and resale liquidity at the cost of area. A third may be financially attractive but require renovation. The Pareto frontier preserves these efficient alternatives before the system asks for a final synthesis. The advocate-versus-critic challenge similarly preserves dissent. A recommendation becomes stronger not because objections disappear, but because the important objections are visible and can be judged against the buyer's objectives.

The interest-rate experiment demonstrates another necessary characteristic of serious decision systems: robustness. Recommendations are conditional on assumptions. A property that leads at one financing rate may become unattractive at another. The correct response is not to hide this instability but to expose it. The same framework can later test holding periods, down payments, renovation shocks, insurance costs, appreciation assumptions, family changes and alternative commute scenarios.

The most important extension, however, is not another data source or another agent. It is **learning the buyer through revealed preference**. The initial conversation can only approximate the client's true objective function. Once the buyer begins interacting with real opportunities, behavior becomes new evidence. Saving a property, rejecting another, repeatedly returning to a neighborhood, accepting an older building in exchange for walkability, or abandoning a larger home because of commute are observations about utility. A production system should update the Buyer Mandate continuously from those choices.

At that point the architecture becomes more than a search engine. It becomes an adaptive relationship between a person and a society of artificial specialists. The system listens, formalizes, searches, deliberates, challenges, governs, explains and then learns. Public and authorized private data feeds can replace the synthetic laboratory without changing this fundamental architecture. Images can add multimodal architectural analysis; maps can add travel-time reasoning; comparable sales can improve valuation; documents can support diligence; and persistent memory can make each subsequent conversation more informed than the last.

The broader research proposition extends well beyond housing. Many important decisions in finance and management have the same structure: the objective is partially latent, evidence is heterogeneous, specialists disagree, constraints matter, uncertainty is unavoidable and the final decision must remain explainable. Real estate gives us an unusually intuitive setting in which to build and test that architecture.

The central achievement of the notebook is therefore not a claim that fifty artificial agents can identify the objectively best house. There may be no such house. The achievement is a framework for transforming an ambiguous human objective into a governed, plural, auditable and adaptive decision process. That is a much more interesting conception of what an intelligent application can become.